# Document Q&A (RAG) — демо
**1)** тесттер → **2)** іздеу (кілтсіз!) → **3)** өз мәтініңді қос → **4)** ИИ жауап береді.

## 1. Жобаны жүктеу және тесттер

In [ ]:
import os
os.chdir("/content")
!rm -rf doc-qa-rag && git clone -q https://github.com/dos1k77/doc-qa-rag
%cd /content/doc-qa-rag
!python -m unittest -v

## 2. Іздеу демосы (кілт керек емес)
Құжаттағы ең ұқсас үзінділерді BM25 арқылы табады. `queries` тізімін өзгертіп көр.

In [ ]:
from rag import BM25, load_chunks

chunks = load_chunks("docs")
index = BM25([text for _, text in chunks])
print(f"{len(chunks)} бөлік индекстелді\n")

queries = ["What is RAG?", "RAG деген не?", "BM25"]
for q in queries:
    print("Сұрақ:", q)
    for score, i in index.top_k(q, k=2):
        src, text = chunks[i]
        print(f"  [{score:.2f}] {src}: {text[:110]}...")
    print()

## 3. Өз мәтініңді қос
`my_text` ішіне кез келген мәтін қой (мақала, конспект, ереже) және ұяшықты іске қос.

In [ ]:
my_text = """
Мұнда өз мәтініңді қой. Мысалы: Абай Құнанбайұлы 1845 жылы туған. Ол қазақтың ұлы ақыны және ойшылы.
"""
with open("docs/my.txt", "w", encoding="utf-8") as f:
    f.write(my_text)
chunks = load_chunks("docs")
index = BM25([text for _, text in chunks])
print("Енді", len(chunks), "бөлік бар")

## 4. ИИ жауап береді (Gemini кілті керек)
Кілт: https://aistudio.google.com/apikey

In [ ]:
import os, getpass
os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key: ")
print("OK, кілт енгізілді (ол GitHub-қа түспейді).")

In [ ]:
from ask import answer

question = "Абай қай жылы туған?"   # <- өз сұрағыңды жаз
text, passages = answer(question, chunks, index)
print(text)
print()
for i, (src, _) in enumerate(passages, 1):
    print(f"[{i}] {src}")